# Gold — Preços Mensais por Município

Este notebook executa e valida a construção da camada Gold de preços mensais de combustíveis por município.

Os dados Silver da ANP são integrados à dimensão oficial de municípios do IBGE e agregados mensalmente.

A granularidade do produto analítico é:

**ano_mes + codigo_ibge + produto**

A Gold é publicada em formato Parquet e particionada por ano.

In [12]:
from pathlib import Path

import polars as pl

from insightfuel_data_platform.ingestion.ibge import (
    baixar_municipios_ibge,
)
from insightfuel_data_platform.pipelines.anp import (
    construir_gold_precos_mensais,
)
from insightfuel_data_platform.transformation.ibge import (
    transformar_municipios_ibge,
)
from insightfuel_data_platform.validation.gold import (
    validar_gold_precos_mensais,
)

## 1. Configuração

São definidos os diretórios de entrada da camada Silver e de publicação da Gold.

In [13]:
pasta_silver = Path(
    "../data/silver/anp/automotivos"
)

pasta_gold = Path(
    "../data/gold/anp/precos_mensais_municipio"
)

## 2. Dimensão geográfica

A dimensão oficial de municípios do IBGE fornece o código geográfico utilizado como chave de integração da camada analítica.

In [14]:
df_ibge = baixar_municipios_ibge()

df_municipios = transformar_municipios_ibge(
    df_ibge
)

print(df_municipios.shape)
print(df_municipios.null_count())

(5571, 5)
shape: (1, 5)
┌─────────────┬───────────┬─────┬────────┬─────────────────┐
│ codigo_ibge ┆ municipio ┆ uf  ┆ regiao ┆ municipio_match │
│ ---         ┆ ---       ┆ --- ┆ ---    ┆ ---             │
│ u32         ┆ u32       ┆ u32 ┆ u32    ┆ u32             │
╞═════════════╪═══════════╪═════╪════════╪═════════════════╡
│ 0           ┆ 0         ┆ 0   ┆ 0      ┆ 0               │
└─────────────┴───────────┴─────┴────────┴─────────────────┘


## 3. Construção da Gold

O pipeline consolida as partições Silver da ANP, realiza a integração geográfica, agrega os preços mensalmente, valida o produto analítico e publica as partições anuais da Gold.

In [15]:
caminhos_publicados = construir_gold_precos_mensais(
    pasta_silver,
    pasta_gold,
    df_municipios,
)

for caminho in caminhos_publicados:
    print(caminho)

../data/gold/anp/precos_mensais_municipio/ano=2023/dados.parquet
../data/gold/anp/precos_mensais_municipio/ano=2024/dados.parquet
../data/gold/anp/precos_mensais_municipio/ano=2025/dados.parquet


## 4. Validação da Gold publicada

As partições publicadas são lidas novamente para verificar a integridade do produto analítico persistido em Parquet.

In [16]:
arquivos_gold = sorted(
    pasta_gold.rglob("*.parquet")
)

df_gold = pl.concat([
    pl.read_parquet(arquivo)
    for arquivo in arquivos_gold
])

validar_gold_precos_mensais(
    df_gold
)

print("Gold publicada válida.")
print("Linhas:", df_gold.height)
print("Partições:", len(arquivos_gold))

Gold publicada válida.
Linhas: 77011
Partições: 3


## 5. Inspeção do produto analítico

In [17]:
df_gold.head(10)

ano_mes,codigo_ibge,municipio,uf,regiao,produto,preco_medio,preco_mediano,preco_minimo,preco_maximo,desvio_padrao,qtd_coletas
date,str,str,str,str,str,f64,f64,f64,f64,f64,u32
2023-01-01,"""1100023""","""Ariquemes""","""RO""","""Norte""","""DIESEL""",7.09275,7.16,6.48,7.49,0.233831,40
2023-01-01,"""1100023""","""Ariquemes""","""RO""","""Norte""","""DIESEL S10""",7.096857,7.18,6.52,7.46,0.226349,35
2023-01-01,"""1100023""","""Ariquemes""","""RO""","""Norte""","""ETANOL""",4.955714,4.99,4.79,5.17,0.171548,7
2023-01-01,"""1100023""","""Ariquemes""","""RO""","""Norte""","""GASOLINA""",5.279333,5.19,4.99,6.37,0.260501,45
2023-01-01,"""1100023""","""Ariquemes""","""RO""","""Norte""","""GASOLINA ADITIVADA""",5.4425,5.36,5.09,6.37,0.34101,36
2023-01-01,"""1100049""","""Cacoal""","""RO""","""Norte""","""DIESEL""",7.084286,7.095,6.7,7.39,0.216477,28
2023-01-01,"""1100049""","""Cacoal""","""RO""","""Norte""","""DIESEL S10""",7.138214,7.13,6.72,7.49,0.238033,28
2023-01-01,"""1100049""","""Cacoal""","""RO""","""Norte""","""ETANOL""",4.886154,4.99,4.56,4.99,0.186259,13
2023-01-01,"""1100049""","""Cacoal""","""RO""","""Norte""","""GASOLINA""",5.4365625,5.45,5.31,5.6,0.057899,32


In [18]:
print(df_gold.schema)

Schema({'ano_mes': Date, 'codigo_ibge': String, 'municipio': String, 'uf': String, 'regiao': String, 'produto': String, 'preco_medio': Float64, 'preco_mediano': Float64, 'preco_minimo': Float64, 'preco_maximo': Float64, 'desvio_padrao': Float64, 'qtd_coletas': UInt32})


## 6. Resultado

A Gold de preços mensais consolida as observações da ANP em uma granularidade mensal por município e produto.

O produto contém métricas de preço médio, mediano, mínimo, máximo, desvio-padrão e quantidade de coletas.

Valores nulos de desvio-padrão são permitidos quando existe apenas uma observação no grupo, pois nesse caso não há observações suficientes para o cálculo do desvio-padrão amostral.

A camada é publicada em Parquet e particionada por ano, estando pronta para consumo analítico e para enriquecimentos posteriores com dados socioeconômicos, geográficos e macroeconômicos.